### Vector stores and retrievers
This video tutorial will familiarize you with LangChain's vector store and retriever abstractions.  These abstractions are designed to support retrieval of data-- from (vector) databases and other sources-- for
integration with LLM workflows.  
They are important for applications that fetch data to be reasoned over as part of model inference, as in the case of retrieval-augmented generation.

We will cover:
- Documents
- Vector stores
- Retrievers

#### Documents
LangChain implements a Document abstraction, which is intended to represent a unit of text and associated metadata. It has two attributes:
- page_content: a string representing the content;
- metadata: a dict containing arbitrary metadata.
  
The metadata attribute can capture information about the source of the document, its relationship to other documents, and other information. Note that an individual Document object often represents a chunk of a larger document.  
  
Let's generate some sample documents:

In [1]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source": "mammal-pets-doc"}
    ),
    Document(
        page_content="Cats are independent pets that often enjoy their own space.",
        metadata={"source": "mammal-pets-doc"}
    ),
    Document(
        page_content="Goldfish are popular pets for beginners, requiring relatively simple care.",
        metadata={"source": "fish-pets-doc"}
    ),
    Document(
        page_content="Parrots are intelligent birds capable of mimicking human speech.",
        metadata={"source": "bird-pets-doc"}
    ),
    Document(
        page_content="Rabbits are social animals that need plenty of space to hop around.",
        metadata={"source": "mammal-pets-doc"}
    ),
]

d:\Agentic AI 101\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['OPENROUTER_API_KEY'] = os.getenv("OPENROUTER_API_KEY") or ""
os.environ['HF_TOKEN'] = os.getenv("HUGGINGFACE_TOKEN") or ""
os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGSMITH_API_KEY") or ""
os.environ["LANGSMITH_TRACING"]=os.getenv("LANGSMITH_TRACING") or ""
os.environ["LANGSMITH_PROJECT"]=os.getenv("LANGSMITH_PROJECT") or ""

In [3]:
from langchain_openrouter import ChatOpenRouter

llm = ChatOpenRouter(model="stepfun/step-3.5-flash:free")
llm

ChatOpenRouter(profile={'max_input_tokens': 256000, 'max_output_tokens': 256000, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<openrouter.sdk.OpenRouter object at 0x000001C592F2A7B0>, openrouter_api_key=SecretStr('**********'), app_url='https://docs.langchain.com/oss', app_title='langchain', model_name='stepfun/step-3.5-flash:free', model_kwargs={})

In [4]:
from pyexpat import model

from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
embeddings

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [5]:
# Vector Stores (vs)
from langchain_chroma import Chroma

vs = Chroma.from_documents(documents, embeddings)
vs


In [6]:
vs.similarity_search("cat")

[Document(id='53fd7b2b-102c-491e-9b4e-d96604e6cdbc', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='3fc55bf9-09cf-41ee-bf33-11e661b0555e', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='1f61cfc0-9c30-4dc9-8591-1078717c801c', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.'),
 Document(id='eec2c0b9-07d4-4913-9aae-6f896cc789cc', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.')]

#### Async Query

In [7]:
await vs.asimilarity_search("cat")

[Document(id='53fd7b2b-102c-491e-9b4e-d96604e6cdbc', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='3fc55bf9-09cf-41ee-bf33-11e661b0555e', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='1f61cfc0-9c30-4dc9-8591-1078717c801c', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.'),
 Document(id='eec2c0b9-07d4-4913-9aae-6f896cc789cc', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.')]

In [8]:
vs.similarity_search_with_score("cat")

[(Document(id='53fd7b2b-102c-491e-9b4e-d96604e6cdbc', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
  0.9351058602333069),
 (Document(id='3fc55bf9-09cf-41ee-bf33-11e661b0555e', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
  1.574089765548706),
 (Document(id='1f61cfc0-9c30-4dc9-8591-1078717c801c', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.'),
  1.595690369606018),
 (Document(id='eec2c0b9-07d4-4913-9aae-6f896cc789cc', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.'),
  1.6657922267913818)]

### Retrivers
LangChain VectorStore objects do not subclass Runnable, and so cannot immediately be integrated into LangChain Expression Language chains. 
    
LangChain Retrievers are Runnables, so they implement a standard set of methods (e.g., synchronous and asynchronous invoke and batch operations) and are designed to be incorporated in LCEL chains.  
  
We can create a simple version of this ourselves, without subclassing Retriever. If we choose what method we wish to use to retrieve documents, we can create a runnable easily. Below we will build one around the
similarity_search method:

In [9]:
from typing import List

from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

retriever = RunnableLambda(vs.similarity_search).bind(k=1)
retriever.batch(["cat", "dog"])

[[Document(id='53fd7b2b-102c-491e-9b4e-d96604e6cdbc', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='3fc55bf9-09cf-41ee-bf33-11e661b0555e', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

Vectorstores implement an as_retriever method that will generate a Retriever, specifically a VectorStoreRetriever. These retrievers include specific search_type and search_kwargs attributes that identify
what methods of the underlying vector store to call, and how to parameterize them. For instance, we can
replicate the above with the following:

In [10]:
retriever = vs.as_retriever(
    search_type = "similarity",
    search_kwargs = {"k" : 1}
)
retriever.batch(["cat", "dog"])

[[Document(id='53fd7b2b-102c-491e-9b4e-d96604e6cdbc', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='3fc55bf9-09cf-41ee-bf33-11e661b0555e', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

### RAG - Basic Example

In [11]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

# Prompt template
message = """
Answer this question using the provided context only.

{question}

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("human", message)
])

# RAG chain
rag_chain = {
    "context": retriever,
    "question": RunnablePassthrough()
} | prompt | llm

# Invoke
response = rag_chain.invoke("tell me about dogs")

print(response.content)

Based on the provided context: Dogs are great companions, known for their loyalty and friendliness.
